## To do

1. Lier les fonctions ensemble pour trouver un moyen de les update dès qu'un des widgets change
- **fonction interact vs interactive vs interactive_output?** :
https://ipywidgets.readthedocs.io/en/latest/examples/Using%20Interact.html

- Demo 2, Ctrl+F interact pour avoir l'exemple du décorateur @interact
https://colab.research.google.com/drive/1xTu1JCRPzeh6FmXrTeWi8RUsXJa6GxLQ?usp=sharing


2. Implémenter la partie "playoffs" avec un slider (en ce moment, seulement saqison régulière fonctionne)

3. Importer nhl_rink.png pour la partie visualisation pour remplacer la patinoire dessinée

In [ ]:
# Setup
from ift6758.data import (
    get_game, regular_season_game_id, playoff_game_id,
)

import json
import ipywidgets as widgets
from ipywidgets import interact, interactive, interactive_output, IntSlider
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# Widget pour choisir la saison
season_widget = widgets.Dropdown(
    options=[i for i in range(2016,2024)],
    value = 2017,
    description='Season start:',
    disabled=False,
)

# Widget pour choisir le game_id d'une partie en saison régulière
reg_number_widget = widgets.IntSlider(
    description = "Game #",
    value = 1,
    min = 1,
    step = 1,
    max = 1500,
    continuous_update=False
)

# Widget pour choisir le numéro d'événement
event_id_widget = widgets.IntSlider(
    description = "Event #",
    value = 1,
    min = 1,
    step = 1,
    max = 400,
    continuous_update=False
)

# Widget pour choisir toggle playoffs ou non
playoffs_widget = widgets.Checkbox(
    value=False,
    description='Playoffs',
    disabled=False,
    indent=False
)

In [ ]:
# Changer la saison, update en temps réel
@interact(i= season_widget)
def inspect_season(i):
    season = i
    print(f"Season {season}-{season+1}")

# Playoffs on/off, update en temps réel
@interact(j=playoffs_widget)
def playoffs_toggle(j):
    if j == True:
        game_type = 3
    else:
        game_type = 2

    if game_type == 2:
        pass
    # placeholder, doit basculer vers recherche du game ID des playoffs
    else:
        print('playoffs')

# Changer le game ID (saison régulière), update en temps réel
@interact(k = reg_number_widget)
def inspect_reg_game(k):
    game_id = regular_season_game_id(season=season_widget.value, number=k)
    game = get_game(game_id)
    print("Game ID:", game["id"])
    print("Date:", game["gameDate"])
    print("Match:", game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"])

# Dessiner la patinoire (temporaire)
def draw_rink(fig):
    # A 200-foot by 85-foot rectangle.
    fig.add_shape(type="rect", x0=-100, x1=100, y0=-40, y1=40,
                  line_color="#7f95aa")
    for x, colour in [(-25, "#4497e5"), (0, "#cf4545"), (25, "#4497e5")]:
        fig.add_vline(x=x, line_color=colour)
    fig.update_xaxes(range=[-103, 103], title="x (ft)", constrain="domain")
    fig.update_yaxes(range=[-46, 46], title="y (ft)", scaleanchor="x", scaleratio=1)
    fig.update_layout(template="plotly_white", height=460,
                      margin=dict(l=55, r=25, t=55, b=50))
    return fig

# Visualiser l'événement ==> nhl_rink.png à ajouter avec plt.imread('nhl_rink.png)
@interact(l=event_id_widget)
def inspect_event(l):
    # ICI: comment réutiliser le dernier game_ID??
    game_id = regular_season_game_id(season=season_widget.value, number=reg_number_widget.value)
    game = get_game(game_id)
    plays = game["plays"]
    play = plays[l]
    details = play.get("details", {})
    x = details.get("xCoord")
    y = details.get("yCoord")
    fig = go.Figure()
    # Plot only if both coordinates exist.
    if x is not None and y is not None:
        fig.add_scatter(x=[x], y=[y], mode="markers", marker_size=18)
    fig.update_layout(title=f"gameID: {game['id']} · eventId : {play['eventId']} · {play['typeDescKey']} · minute {play['timeInPeriod']} · period {play['periodDescriptor']['number']}")
    draw_rink(fig).show()

interactive(children=(Dropdown(description='Season start:', index=1, options=(2016, 2017, 2018, 2019, 2020, 20…

interactive(children=(Checkbox(value=False, description='Playoffs', indent=False), Output()), _dom_classes=('w…

interactive(children=(IntSlider(value=1, continuous_update=False, description='Game #', max=1500, min=1), Outp…

interactive(children=(IntSlider(value=1, continuous_update=False, description='Event #', max=400, min=1), Outp…

In [ ]:
# Exemple de la fonction interactive_output

a = widgets.IntSlider(description='a')
b = widgets.IntSlider(description='b')
c = widgets.IntSlider(description='c')
def f(a, b, c):
    print('{}*{}*{}={}'.format(a, b, c, a*b*c))

out = widgets.interactive_output(f, {'a': a, 'b': b, 'c': c})

widgets.HBox([widgets.VBox([a, b, c]), out])

In [ ]:
# matplotlib (plt.) avec le png au lieu de tracer l'image ()
# pas encore testé vu que je n'arrivais pas à récupérer l'image, je pourrai m'en occuper
"""
img = plt.imread()
    # Plot only if both coordinates exist.
    if x is not None and y is not None:
        plt.scatter(x=[x], y=[y], mode="markers", marker_size=18)
    plt.imshow(img, zorder=0, extent=[0.5, 8.0, 1.0, 7.0])
    plt.show()
"""